In [28]:
import xml.etree.ElementTree as ET
from lxml import etree, html

In [ ]:
parser = etree.HTMLParser()

tree = html.parse('./data/index_diachronica.html')
body = tree.getroot().find('body')

document = etree.Element('document')
document

to_super = str.maketrans("0123456789","⁰¹²³⁴⁵⁶⁷⁸⁹")
to_sub = str.maketrans("0123456789","₀₁₂₃₄₅₆₇₈₉")

for s in body.iter('section'):
    h2 = s.find('h2')
    idx, name = h2.text.split(' ', 1)

    section = etree.Element('section')
    section.set('index', idx)
    section.set('name', name)

    document.append(section)

    lines = s.findall('p')

    for idx, line in enumerate(lines):
        author = line.find('i')
        if idx == 0 and len(line.attrib) == 0 and author is not None:
            cite = etree.Element('cite')
            cite.text = line.text_content().strip()
            section.append(cite)
        else:
            if 'rule' == line.get('class', ''):
                el = etree.Element('rule')
            else:
                el = etree.Element('comment')

            sub = line.findall('sub')
            sup = line.findall('sup')

            for s in sub:
                s.text = s.text.translate(to_sub)
            for s in sup:
                s.text = s.text.translate(to_super)

            el.text = line.text_content().strip()
            section.append(el)
                      

with open("./data/index_diachronica.xml", 'wb') as out: 
    xml = etree.tostring(document, encoding='UTF-8', method='xml', pretty_print=True)
    out.write(xml)
